# Solutions to project 3: a four-vector class that runs the whole analysis

In project 2 a four-momentum was a dictionary, and three separate functions worked on it.
Here you write a class that keeps the data and the methods together, and then a class that runs the whole analysis, from settings to histogram.
The physics is the one from project 1: four leptons carry the mass of the Higgs that made them.

<img src="../img/class-objects.svg" width="80%" alt="">

**How to work**

- Run each cell with `Shift` + `Enter`, from top to bottom.
- Replace every `...` with your own code.

In [ ]:
import hist
import numpy as np

import awkward as ak  # only used here, to read the file into NumPy arrays

raw = ak.from_parquet("../data/SMHiggsToZZTo4L.parquet")
raw = raw[(ak.num(raw.electron) == 2) & (ak.num(raw.muon) == 2)]
fields = ["pt", "eta", "phi", "mass", "charge"]
events = {
    name: {field: ak.to_numpy(raw[collection][field][:, index]) for field in fields}
    for name, collection, index in [("e1", "electron", 0), ("e2", "electron", 1), ("mu1", "muon", 0), ("mu2", "muon", 1)]
}
e1, e2, mu1, mu2 = events["e1"], events["e2"], events["mu1"], events["mu2"]

`events` holds the same 30,868 events as project 2: one dictionary per lepton, one array per field.

## Part 1: `__init__`

`__init__` stores the data on a new object.

The cell below holds the whole class, with three gaps.
Complete one method at a time, run the class cell again, then run the part's cell below.

**Task.** Complete `__init__` so that it stores all four components.

In [ ]:
class FourVector:
    def __init__(self, E, px, py, pz):
        self.E = E
        self.px = px
        self.py = py
        self.pz = pz

    def __add__(self, other):
        return FourVector(
            self.E + other.E,
            self.px + other.px,
            self.py + other.py,
            self.pz + other.pz,
        )

    @property
    def mass(self):
        return np.sqrt(self.E**2 - self.px**2 - self.py**2 - self.pz**2)

In [ ]:
v = FourVector(10.0, 1.0, 2.0, 3.0)
v

## Part 2: `__add__`

`__add__` makes `+` work.

**Task.** Complete `__add__`: it returns a new `FourVector` whose four components are the sums of the two.

In [ ]:
w = FourVector(10.0, 1.0, 2.0, 3.0) + FourVector(5.0, 0.0, -1.0, 1.0)
w

## Part 3: Properties

A property is computed when you read it.

**Task.** Complete `mass`: $m = \sqrt{E^2 - p_x^2 - p_y^2 - p_z^2}$, with `np.sqrt` so that it also works on arrays.

In [ ]:
FourVector(5.0, 1.0, 2.0, 2.0).mass

## Part 4: Leptons as four-vectors

One function turns a lepton into a `FourVector`.

**Task.** Complete `lepton_vector`. The formulas are the ones from projects 1 and 2.

In [ ]:
def lepton_vector(lepton):
    px = lepton["pt"] * np.cos(lepton["phi"])
    py = lepton["pt"] * np.sin(lepton["phi"])
    pz = lepton["pt"] * np.sinh(lepton["eta"])
    E = np.sqrt(px**2 + py**2 + pz**2 + lepton["mass"] ** 2)
    return FourVector(E, px, py, pz)


z_ee = lepton_vector(e1) + lepton_vector(e2)
z_ee.mass

## Part 5: One class runs it all

A class holds the settings and the work.

`__init__` stores the Z mass windows. `run` does the analysis of project 2 on some events and returns a dictionary of results.

**Task.** Fill in `opposite`, `z_mumu` and what to fill into the histogram.

In [ ]:
class HiggsAnalysis:
    def __init__(self, z_small=(12, 120), z_big=(40, 120)):
        self.z_small = z_small
        self.z_big = z_big

    def run(self, events):
        e1, e2, mu1, mu2 = events["e1"], events["e2"], events["mu1"], events["mu2"]
        opposite = (e1["charge"] != e2["charge"]) & (mu1["charge"] != mu2["charge"])
        z_ee = lepton_vector(e1) + lepton_vector(e2)
        z_mumu = lepton_vector(mu1) + lepton_vector(mu2)
        small = np.minimum(z_ee.mass, z_mumu.mass)
        big = np.maximum(z_ee.mass, z_mumu.mass)
        in_small = (self.z_small[0] < small) & (small < self.z_small[1])
        in_big = (self.z_big[0] < big) & (big < self.z_big[1])
        selected = opposite & in_small & in_big

        h = hist.Hist.new.Reg(50, 80, 180, label="four-lepton mass [GeV]").Double()
        h.fill((z_ee + z_mumu).mass[selected])
        return {"events": len(selected), "selected": int(selected.sum()), "mass": h}


output = HiggsAnalysis().run(events)
output["selected"]

In [ ]:
output["mass"].plot();

## Part 6: Adding results

Results from separate pieces of data add up.

`take` below slices out a range of events, with the dictionary comprehension from lesson 1.
Run the analysis on each half, then combine: the sum must equal the all-at-once answer.

**Task.** Combine the two halves' results, key by key. Numbers and histograms both add with `+`.

In [ ]:
def take(events, start, stop):
    """The slice [start:stop] of every array in events."""
    return {lepton: {field: values[start:stop] for field, values in arrays.items()} for lepton, arrays in events.items()}


half = len(events["e1"]["pt"]) // 2
analysis = HiggsAnalysis()
first = analysis.run(take(events, 0, half))
second = analysis.run(take(events, half, None))
total = {key: first[key] + second[key] for key in first}

total["selected"]

In [ ]:
total["mass"].plot();

Two halves give the same answer as all events at once.

## Bonus: Inheritance

Inheritance changes only what differs.

**Task.** Write `TightHiggsAnalysis`, which inherits from `HiggsAnalysis` and only changes the windows: the lighter Z between 20 and 120 GeV, the heavier between 60 and 120 GeV.
Use `super().__init__(...)`.

In [ ]:
class TightHiggsAnalysis(HiggsAnalysis):
    def __init__(self):
        super().__init__(z_small=(20, 120), z_big=(60, 120))


TightHiggsAnalysis().run(events)["selected"]

**Done.** Settings, data and code now travel together: one object runs the whole analysis, and two results combine with one line. Every tool you import is built this way.